# THLP MC Benchmark

**Track:** Trinity Hierarchical Learning Pattern
**Dataset:** Adversarial (274 questions)
**Framework:** Claude API

This notebook evaluates the THLP cognitive track using adversarial questions to test true pattern learning capabilities.

In [ ]:
# Install dependencies
!pip install -q anthropic pandas numpy

In [ ]:
import os
import pandas as pd
import numpy as np
from anthropic import Anthropic

# Set your API key
os.environ['ANTHROPIC_API_KEY'] = 'your-api-key-here'

client = Anthropic(api_key=os.environ.get('ANTHROPIC_API_KEY'))

print("THLP Benchmark initialized")

In [ ]:
# Load dataset
df = pd.read_csv('/kaggle/input/trinity-cognitive-probes-thlp-mc/thlp_mc.csv')

print(f"Loaded {len(df)} questions")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst row:\n{df.iloc[0].to_dict()}")

In [ ]:
def evaluate_question(question, choices, correct_answer):
    """Evaluate a single question using Claude."""
    prompt = f"""Answer the following multiple choice question with ONLY ONE letter (A, B, C, or D).

Question: {question}

Choices:
A: {choices.get('A', '')}
B: {choices.get('B', '')}
C: {choices.get('C', '')}
D: {choices.get('D', '')}

Your answer (A/B/C/D):"""
    
    try:
        response = client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=10,
            messages=[{"role": "user", "content": prompt}]
        )
        answer = response.content[0].text.strip().upper()[0]
        return answer if answer in ['A', 'B', 'C', 'D'] else 'A'
    except Exception as e:
        print(f"Error: {e}")
        return 'A'

In [ ]:
# Run evaluation (sample first 10 for testing)
results = []
sample_df = df.head(10)

for idx, row in sample_df.iterrows():
    question = row['question']
    choices = {
        'A': row.get('A', ''),
        'B': row.get('B', ''),
        'C': row.get('C', ''),
        'D': row.get('D', '')
    }
    correct_answer = row['answer'].strip().upper()
    
    predicted = evaluate_question(question, choices, correct_answer)
    is_correct = predicted == correct_answer
    
    results.append({
        'id': row.get('id', idx),
        'question': question[:50] + '...',
        'predicted': predicted,
        'correct': correct_answer,
        'is_correct': is_correct
    })

results_df = pd.DataFrame(results)
accuracy = results_df['is_correct'].mean()

print(f"\nTHLP Accuracy (sample): {accuracy:.2%}")
print(f"\nResults:\n{results_df}")

In [ ]:
# Save submission
submission = df[['id']].copy()
submission['prediction'] = 'A'  # Replace with actual predictions
submission.to_csv('/kaggle/working/submission_thlp.csv', index=False)

print("Submission saved to /kaggle/working/submission_thlp.csv")